# LAB S05: LLMs para extracción estructurada

Cada sección usa el archivo de `mi_pipeline` que se escribe en su paso de la guía; el cuaderno solo importa y ejecuta. El centro de la sesión es el prompt: Rol, Contexto, Tarea y Formato, y un ejemplo resuelto; el cuaderno lo imprime tal como lo recibe el modelo. Los modelos corren en esta VM con Ollama: ningún documento sale a servicios en la nube. Los documentos son ficticios y cada uno trae su respuesta correcta para medir.

## 0. Dónde están los datos de práctica

In [ ]:
# Celda estándar de localización de datos (no se modifica)
import os
from pathlib import Path


def localizar_carpeta(nombre="02_Datos_Practica", max_niveles=6, obligatorio=True):
    """Busca `nombre` en la carpeta actual y en sus carpetas superiores."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        candidata = carpeta / nombre
        if candidata.is_dir():
            return candidata
    # Respaldo: carpetas del curso en la unidad del sistema (VM de laboratorio)
    curso = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso"
    respaldo = [curso / "paquete" / nombre]
    if nombre == "02_Datos_Practica":
        respaldo.insert(0, curso / "datos")
    for candidata in respaldo:
        if candidata.is_dir():
            return candidata
    if obligatorio:
        raise FileNotFoundError(
            f"No se encuentra la carpeta '{nombre}'. El cuaderno se abre desde su carpeta "
            "dentro del repositorio del curso, sin moverlo de lugar."
        )
    return None


DATOS = localizar_carpeta("02_Datos_Practica")
print("Datos de práctica encontrados:", DATOS.name)
print("Subcarpetas:", sorted(p.name for p in DATOS.iterdir() if p.is_dir()))


## Paso 0: conexión con `mi_pipeline` y rama de trabajo (C0)

In [ ]:
# Conexión de mi_pipeline y de las herramientas del LAB (no se modifica)
%load_ext autoreload
%autoreload 2
import html
import json
import re
import shutil
import subprocess
import sys
import tempfile
import time
import unicodedata
import warnings

import pandas as pd
import pymupdf
import pytesseract
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)   # avisos de cambios futuros de las bibliotecas, que no cambian el resultado
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)


def localizar_raiz(max_niveles=6):
    """Carpeta del repositorio clonado que contiene mi_pipeline."""
    if os.environ.get("MI_PIPELINE_RAIZ"):
        return Path(os.environ["MI_PIPELINE_RAIZ"])
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        if (carpeta / "mi_pipeline" / "__init__.py").is_file():
            return carpeta
    clon = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso" / "trabajo" / "entorno-extraccion-documental"
    if (clon / "mi_pipeline" / "__init__.py").is_file():
        return clon
    raise FileNotFoundError("No se encuentra la carpeta mi_pipeline: la preparación de la sesión la repone.")


RAIZ = localizar_raiz()
sys.path.insert(0, str(RAIZ))
SALIDAS = Path("salidas")
SALIDAS.mkdir(exist_ok=True)
CONTROLES = []
INICIO = time.perf_counter()
pytesseract.pytesseract.tesseract_cmd = (
    os.environ.get("TESSERACT_CMD")
    or shutil.which("tesseract")
    or str(Path(os.environ.get("ProgramFiles", "")) / "Tesseract-OCR" / "tesseract.exe")
)


def correr_pruebas():
    """Corre las pruebas de mi_pipeline con pytest, muestra el resumen y dice si todas pasaron."""
    corrida = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider",
         "--basetemp", tempfile.mkdtemp(prefix="pytest_s05_"), str(RAIZ / "mi_pipeline" / "tests")],
        capture_output=True, text=True, cwd=str(RAIZ), timeout=900, encoding="utf-8", errors="replace",
    )
    resumen = [linea for linea in corrida.stdout.strip().splitlines() if " passed" in linea or " failed" in linea or "error" in linea]
    print(re.sub(r" in [\d.]+s.*", "", resumen[-1]) if resumen else corrida.stdout[-500:])
    return corrida.returncode == 0


# Cómo se muestran los resultados: tablas alineadas, con el texto completo y los números como se escriben en el Perú
PORCENTAJES = {"exactitud", "precision", "exhaustividad", "f1", "glm-ocr", "Tesseract"}
SIN_MILES = {"contexto", "nivel"}


def num(valor, decimales=2):
    """Un número con punto de miles, coma decimal y sin ceros de más: 7.728 o 10,78."""
    texto = f"{valor:,.{decimales}f}".replace(",", "§").replace(".", ",").replace("§", ".")
    return texto.rstrip("0").rstrip(",") if "," in texto else texto


def pct(valor):
    """Una proporción como porcentaje: 95,8 %."""
    return f"{valor * 100:.1f} %".replace(".", ",")


def si_no(valor):
    return "sí" if valor else "no"


def como_texto(valor, columna="", vacio="vacío"):
    """Un valor tal como se muestra en una tabla: sí o no, números en formato local y «vacío» si no hay dato."""
    if pd.api.types.is_scalar(valor) and pd.isna(valor):
        return vacio
    if hasattr(valor, "item"):
        valor = valor.item()
    if isinstance(valor, bool):
        return si_no(valor)
    if isinstance(valor, (int, float)):
        if columna in PORCENTAJES:
            return pct(valor)
        if columna in SIN_MILES:
            return str(valor)
        return num(valor, 3 if "CER" in columna else 2)
    return str(valor)


def ver(tabla, titulo, vacio="vacío"):
    """Muestra una tabla alineada y con el texto completo; la tabla original no cambia."""
    if tabla.empty:
        print(f"{titulo}. Ninguno.")
        return
    base = tabla.reset_index() if tabla.index.name else tabla
    vista = pd.DataFrame({columna: [como_texto(v, str(columna), vacio) for v in base[columna]] for columna in base.columns})
    cuerpo = (vista.to_html(index=False, border=0)
              .replace("<th>", '<th style="text-align:left;vertical-align:bottom;padding:4px 10px">')
              .replace("<td>", '<td style="text-align:left;vertical-align:top;white-space:normal;padding:4px 10px">'))
    display({"text/html": f"<p><b>{html.escape(titulo)}</b></p>{cuerpo}", "text/plain": titulo + "\n" + vista.to_string(index=False)}, raw=True)


def decir(texto):
    """Dice qué mostró esta corrida, con sus propios números."""
    display({"text/html": f'<p style="border-left:4px solid #888;padding:6px 10px"><b>En esta corrida:</b> {html.escape(texto)}</p>',
             "text/plain": "En esta corrida: " + texto}, raw=True)


# Medición contra la respuesta correcta de cada documento (la misma vara en todos los ejercicios)
NO_SE_MIDEN = {"ambiguedad", "cuadra", "paginas", "n_paginas", "monto_en_texto", "canal", "adjunto",
               "n_requisitos", "n_requisitos_cumplen", "n_requisitos_no_cumplen", "n_requisitos_observados"}
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "setiembre", "octubre", "noviembre", "diciembre"]


def llano(texto):
    """Minúsculas, sin tildes y con un solo espacio."""
    base = "".join(c for c in unicodedata.normalize("NFD", str(texto)) if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", base.lower()).strip(" .,:;")


def figura(valor, texto):
    """Dice si una fecha (en cualquiera de sus formas escritas) o un código figura en el texto."""
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor):
        anio, mes, dia = valor.split("-")
        meses = [MESES[int(mes) - 1]] + (["septiembre"] if mes == "09" else [])
        formas = [valor, f"{dia}/{mes}/{anio}", f"{int(dia)}/{mes}/{anio}"] + [f"{d} de {m} de {anio}" for m in meses for d in (dia, int(dia))]
        return any(forma in texto for forma in formas)
    return llano(valor) in texto


def correcto_de(id_):
    """Respuesta correcta de un documento: sus campos, con null en las fechas y códigos que no figuran en su texto."""
    datos = json.loads((DATOS / "10_Ground_Truth" / f"{id_}.json").read_text(encoding="utf-8"))
    texto = llano(datos["texto_completo"])
    correcto = {}
    for campo, valor in datos["campos"].items():
        if campo in NO_SE_MIDEN or isinstance(valor, (bool, dict, list)):
            continue
        if isinstance(valor, str) and (re.fullmatch(r"\d{4}-\d{2}-\d{2}", valor) or campo == "numero_reclamo") and not figura(valor, texto):
            valor = None
        correcto[campo] = valor
    return correcto


def es_correcto(valor, correcto):
    """Medida estricta: el dato listo para usar (número como número, fecha AAAA-MM-DD, moneda PEN); null es correcto si no hay dato."""
    if correcto is None:
        return valor is None
    if valor is None:
        return False
    if isinstance(correcto, (int, float)):
        return isinstance(valor, (int, float)) and not isinstance(valor, bool) and abs(valor - correcto) < 0.011
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}|\d{8,20}", str(correcto)):
        return str(valor).strip() == correcto
    return llano(valor) == llano(correcto)


def aciertos(extraido, correcto):
    """(correctos, esperados, fallos) de un documento."""
    escrito = lambda v: "vacío" if v is None else repr(v)
    fallos = [f"{campo}: {escrito(extraido.get(campo))} en lugar de {escrito(valor)}" for campo, valor in correcto.items() if not es_correcto(extraido.get(campo), valor)]
    return len(correcto) - len(fallos), len(correcto), fallos


print("Carpeta del repositorio:", RAIZ.name, "| Tesseract:", pytesseract.get_tesseract_version())

In [ ]:
# ✔ Control C0: la rama de trabajo
rama = subprocess.run(["git", "branch", "--show-current"], cwd=RAIZ, capture_output=True, text=True).stdout.strip()
assert rama == "trabajo-s05", f"C0: la rama actual es '{rama}'; la preparación de la sesión crea trabajo-s05 desde trabajo-s04."
CONTROLES.append("C0")
print("✔ C0 superado: rama de trabajo", rama)

## Reto: el prompt más corto posible

In [ ]:
# Reto: el prompt más corto posible, con el texto que da un lector básico y nada más
import docx
import ollama

local = ollama.Client(host="http://127.0.0.1:11434", timeout=120)


def texto_simple(ruta):
    """Lo que un lector básico saca del archivo: la capa de texto del PDF, los párrafos del DOCX o nada (imagen)."""
    if ruta.suffix == ".pdf":
        with pymupdf.open(ruta) as pdf:
            return "\n".join(pagina.get_text() for pagina in pdf).strip()
    if ruta.suffix == ".docx":
        return "\n".join(p.text for p in docx.Document(str(ruta)).paragraphs).strip()
    return ""


RETO = [DATOS / "05_Polizas" / "POL-0001.pdf", DATOS / "02_Oficios" / "OFI-0001.pdf",
        DATOS / "08_Vouchers" / "VOU-0001.jpg", DATOS / "04_Contratos" / "CON-0001.docx"]
respuestas_reto = {}
for ruta in RETO:
    texto = texto_simple(ruta)
    mensajes = [{"role": "user", "content": "Extrae los datos de este documento:\n\n" + texto}]
    print(f"===== {ruta.name}: el prompt que recibe el modelo ({len(mensajes)} mensaje) =====")
    for mensaje in mensajes:
        recorte = mensaje["content"][:200] + (" [...]" if len(mensaje["content"]) > 200 else "")
        print(f"papel: {mensaje['role']} | {len(mensaje['content'])} caracteres\n{recorte}")
    r = local.chat(model="qwen3.5:4b", messages=mensajes, think=False,
                   options={"num_ctx": 12288, "num_predict": 300, "temperature": 0, "seed": 2026})
    respuestas_reto[ruta.name] = {"caracteres": len(texto), "respuesta": r.message.content.strip()}
    print("----- respuesta del modelo -----\n" + r.message.content.strip()[:500] + "\n")
esquema_minimo = {"type": "object", "required": ["numero_oficio", "fecha", "entidad", "ruc"],
                  "properties": {c: {"type": ["string", "null"]} for c in ("numero_oficio", "fecha", "entidad", "ruc")}}
con_esquema = local.chat(model="qwen3.5:4b", format=esquema_minimo, think=False, options={"temperature": 0, "seed": 2026, "num_predict": 200},
                         messages=[{"role": "user", "content": "Extrae los datos del oficio. Si un dato no aparece, usa null.\n\nTEXTO:\n" + texto_simple(RETO[1])}])
print("El oficio escaneado con un formato mínimo (un esquema en format) y su texto vacío:", con_esquema.message.content)
ver(pd.DataFrame([{"archivo": nombre, "caracteres que recibió el modelo": datos["caracteres"]} for nombre, datos in respuestas_reto.items()]), "Cuánto texto recibió el modelo de cada archivo")

**Cómo leer esta salida**

- **Qué se ve:** Por cada archivo, el prompt que recibió el modelo y su respuesta; al final, una tabla con los caracteres que recibió de cada archivo.
- **Qué mirar:** Los archivos con 0 caracteres: el modelo no recibió ningún texto de ellos. En los demás, si todas las respuestas tienen la misma forma.
- **Qué se hace con esto:** En la celda siguiente se anotan las causas: ¿qué diferencia a los archivos con 0 caracteres de los demás?, y ¿qué le falta al prompt para que todas las respuestas tengan la misma forma?

In [ ]:
# Causas de las fallas: el participante escribe al menos dos, entre comillas y separadas por coma
causas = []
if len(causas) >= 2:
    print(f"✔ Reto superado: anotó {len(causas)} causas ->", "; ".join(causas))
else:
    print("✘ Falta anotar al menos dos causas en la lista `causas` y volver a correr esta celda")

## Paso 1: `llm/cliente_llm.py` (E1, C1 y C2)

In [ ]:
# ✔ Control C1: Ollama responde y tiene los modelos de la sesión
import ollama

local = ollama.Client(host="http://127.0.0.1:11434", timeout=120)
modelos = sorted(m.model for m in local.list().models)
print("Modelos descargados en esta máquina:", ", ".join(modelos))
faltan = [m for m in ("qwen3.5:4b", "glm-ocr:latest") if m not in modelos]
assert not faltan, f"C1: faltan los modelos {faltan}; la instalación en otra ventana todavía los descarga (o se repite el paso de instalación)."
prueba = local.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Responde solo: listo"}], think=False, options={"num_predict": 5})
print("qwen3.5:4b responde:", prueba.message.content.strip())
CONTROLES.append("C1")
print("✔ C1 superado: Ollama responde con qwen3.5:4b y glm-ocr")

In [ ]:
# E1: el cliente LLM y dos modelos con el mismo prompt sobre 5 pólizas PDF: VRAM, segundos y exactitud (C2)
from mi_pipeline.llm.cliente_llm import configuracion_llm, modelos_cargados, ollama_disponible, preguntar

CFG_LLM = configuracion_llm()
CAMPOS_POLIZA = list(correcto_de("POL-0006"))
PEDIDO_E1 = ("TAREA\nExtraer de la póliza los datos de estos campos: " + ", ".join(CAMPOS_POLIZA) + ". Si un dato no aparece, escribir null.\n\n"
             "FORMATO\nSolo un JSON con esas claves. Fechas en formato AAAA-MM-DD; montos y porcentajes como número; moneda como código: PEN o USD.")
print("Prompt del E1. Mensaje de sistema (solo Tarea y Formato; el Rol y el Contexto llegan en el E3):")
print(PEDIDO_E1)
print("Mensaje del usuario: «DOCUMENTO» y el texto de cada póliza.\n")
filas_e1 = []
for modelo in (CFG_LLM["modelo"], CFG_LLM["modelo_grande"]):
    if not ollama_disponible(modelo):
        print(f"{modelo} no está descargado en esta máquina: la comparación se hace con la VM del compañero.")
        continue
    carga = preguntar(modelo, "Responde solo: listo", num_predict=5)      # la primera llamada sube el modelo a la GPU
    bien = total = 0
    segundos = 0.0
    for numero in range(6, 11):
        id_ = f"POL-{numero:04d}"
        with pymupdf.open(DATOS / "05_Polizas" / f"{id_}.pdf") as pdf:
            texto = pdf[0].get_text()
        mensajes = [{"role": "system", "content": PEDIDO_E1}, {"role": "user", "content": "DOCUMENTO\n" + texto}]
        r = preguntar(modelo, mensajes, esquema="json")
        try:
            extraido = json.loads(r.texto)
        except json.JSONDecodeError:                                  # una respuesta cortada no es JSON: cuenta como vacía
            extraido = {}
        ok, n, _ = aciertos(extraido, correcto_de(id_))
        bien, total, segundos = bien + ok, total + n, segundos + r.segundos
    vram = next(m["vram_gb"] for m in modelos_cargados() if m["modelo"] == modelo)
    filas_e1.append({"modelo": modelo, "VRAM (GB)": vram, "carga (s)": carga.segundos,
                     "segundos por póliza": round(segundos / 5, 1), "correctos": f"{bien}/{total}", "exactitud": round(bien / total, 2)})
tabla_e1 = pd.DataFrame(filas_e1).set_index("modelo")
ver(tabla_e1, "Los modelos con el mismo prompt, sobre 5 pólizas")
print(f"Última llamada: {r.tokens_entrada} tokens de entrada, {r.tokens_salida} de salida, {num(r.segundos)} s; motivo de fin: {r.fin} (stop: el modelo terminó; length: se cortó).")
decir("; ".join(f"{m} ocupa {num(f['VRAM (GB)'], 1)} GB de memoria de video, tarda {num(f['segundos por póliza'], 1)} s por póliza y acierta {f['correctos'].replace('/', ' de ')} campos" for m, f in tabla_e1.iterrows()) + ".")
# ✔ Control C2
assert CFG_LLM["modelo"] in tabla_e1.index and (tabla_e1["VRAM (GB)"] > 0).all(), "C2: la tabla debe traer al menos qwen3.5:4b con su VRAM."
CONTROLES.append("C2")
print("✔ C2 superado: VRAM, segundos y exactitud de", len(tabla_e1), "modelo(s) con el mismo prompt")

**Cómo leer esta salida**

- **Qué es cada fila:** Un modelo, con el mismo prompt sobre las mismas 5 pólizas.
- **Qué dice cada columna:** `VRAM (GB)`: memoria de la tarjeta gráfica que ocupa el modelo. `carga (s)`: segundos para subirlo a la tarjeta; se pagan una sola vez. `segundos por póliza`: lo que tarda cada documento. `correctos`: campos bien extraídos, de 65. `exactitud`: el mismo dato en porcentaje.
- **Qué mirar:** Si los dos modelos aciertan lo mismo, la diferencia está solo en la memoria y en el tiempo.
- **Qué se decide:** Se elige el modelo más pequeño que alcanza la exactitud necesaria: libera memoria y tiempo en la VM.

## Paso 2: `ingesta/leer_documento.py` (E2, C3 y C4)

In [ ]:
# E2: un lector para cada formato; el lector marca lo que necesita OCR (C3)
from mi_pipeline.ingesta.leer_documento import aplicar_ocr, campos_rotulados, leer_documento, texto_de

MUESTRA_E2 = ["06_Reclamos/REC-0001.eml", "05_Polizas/POL-0001.pdf", "02_Oficios/OFI-0001.pdf", "02_Oficios/OFI-0002.pdf",
              "02_Oficios/OFI-0003.pdf", "08_Vouchers/VOU-0001.jpg", "04_Contratos/CON-0001.docx", "07_Tarifarios/TAR-0001.html"]
LEIDOS = {}
filas = []
for rel in MUESTRA_E2:
    doc = leer_documento(DATOS / rel)
    LEIDOS[doc["archivo"]] = doc
    filas.append({"archivo": doc["archivo"], "formato": doc["formato"], "partes": len(doc["partes"]),
                  "caracteres sin OCR": len(texto_de(doc)), "pendiente de OCR": ", ".join(p["origen"] for p in doc["pendientes"]) or "nada"})
tabla_lector = pd.DataFrame(filas).set_index("archivo")
ver(tabla_lector, "Qué trae cada archivo antes del OCR")
sin_texto = [archivo for archivo, f in tabla_lector.iterrows() if f["caracteres sin OCR"] == 0]
decir(f"{len(sin_texto)} de {len(tabla_lector)} archivos no traen ningún texto propio y dependen del OCR: {', '.join(sin_texto)}.")
# ✔ Control C3
assert LEIDOS["OFI-0001.pdf"]["pendientes"] and not LEIDOS["POL-0001.pdf"]["pendientes"], "C3: el oficio escaneado va a OCR y la póliza digital no."
assert any("adjunto" in p["origen"] for p in LEIDOS["REC-0001.eml"]["pendientes"]), "C3: el adjunto del correo debe quedar pendiente de OCR."
CONTROLES.append("C3")
print("✔ C3 superado: el lector marcó", sum(len(d["pendientes"]) for d in LEIDOS.values()), "piezas para OCR")

**Cómo leer esta salida**

- **Qué es cada fila:** Un archivo de la bandeja.
- **Qué dice cada columna:** `formato`: el tipo de archivo. `partes`: páginas o secciones. `caracteres sin OCR`: el texto que el archivo trae por sí solo. `pendiente de OCR`: lo que todavía es una imagen.
- **Qué mirar:** Un 0 en `caracteres sin OCR` significa que el archivo es una imagen o un escaneado: sin OCR, el modelo no recibe nada.
- **Qué se decide:** Todo lo que figura en `pendiente de OCR` pasa por el OCR antes de llegar al modelo.

In [ ]:
# E2: el corte silencioso: glm-ocr con el contexto de serie (4096) frente al del YAML (8192)
correcto_ofi = json.loads((DATOS / "10_Ground_Truth" / "OFI-0001.json").read_text(encoding="utf-8"))["texto_completo"]
corto = aplicar_ocr(LEIDOS["OFI-0001.pdf"], num_ctx=4096)
completo = aplicar_ocr(LEIDOS["OFI-0001.pdf"])
corte, filas = {}, []
for contexto, leido in ((4096, corto), (8192, completo)):
    o = leido["ocr"][0]
    corte[f"contexto {contexto}"] = o
    filas.append({"contexto": contexto, "caracteres que llegaron": o["caracteres"], "caracteres del oficio": len(correcto_ofi),
                  "cómo terminó": "se cortó (length)" if o["fin"] == "length" else "completo (stop)", "segundos": o["segundos"]})
ver(pd.DataFrame(filas), "La misma página con dos contextos")
print("Lo último que llegó con 4096: «…" + " ".join(texto_de(corto)[-70:].split()) + "»")
decir(f"con contexto 4096 llegaron {num(corte['contexto 4096']['caracteres'])} de {num(len(correcto_ofi))} caracteres y la celda no dio error: solo «se cortó» lo delata. Con 8192 llegaron {num(corte['contexto 8192']['caracteres'])}.")
assert corte["contexto 4096"]["fin"] == "length" and corte["contexto 8192"]["caracteres"] > 3 * corte["contexto 4096"]["caracteres"], "Con 4096 la lectura debe cortarse."

**Cómo leer esta salida**

- **Qué es cada fila:** La misma página leída con un contexto distinto. El contexto es lo que cabe a la vista del modelo de una sola vez: la imagen, el prompt y la respuesta. `glm-ocr` también recibe un prompt: «Text Recognition:» y la imagen.
- **Qué dice cada columna:** `caracteres que llegaron`: lo que el modelo alcanzó a escribir. `caracteres del oficio`: lo que tiene el documento. `cómo terminó`: «se cortó» significa que la respuesta llegó al límite; «completo», que el modelo terminó.
- **Qué mirar:** La fila de 4096: llegan pocos caracteres y la celda no da ningún error. El recuadro rosado es un aviso que el propio cuaderno escribe a propósito.
- **Qué se decide:** Toda lectura que «se cortó» se repite con más contexto o se marca para revisión.

In [ ]:
# E2: CER de glm-ocr frente al Tesseract de S03 en los 3 oficios escaneados (C4)
from mi_pipeline.calidad.metricas import cer

filas = []
LECTURAS_TESSERACT = {}
for id_ in ("OFI-0001", "OFI-0002", "OFI-0003"):
    correcto_txt = json.loads((DATOS / "10_Ground_Truth" / f"{id_}.json").read_text(encoding="utf-8"))["texto_completo"]
    glm = completo if id_ == "OFI-0001" else aplicar_ocr(LEIDOS[f"{id_}.pdf"])
    tesseract = LECTURAS_TESSERACT[id_] = aplicar_ocr(LEIDOS[f"{id_}.pdf"], motor="tesseract")
    LEIDOS[f"{id_}.pdf"] = glm                                        # E3 usa esta lectura
    filas.append({"oficio": id_, "CER glm-ocr": round(cer(correcto_txt, texto_de(glm)), 3), "CER Tesseract": round(cer(correcto_txt, texto_de(tesseract)), 3),
                  "segundos glm-ocr": glm["ocr"][0]["segundos"], "segundos Tesseract": tesseract["ocr"][0]["segundos"]})
tabla_cer = pd.DataFrame(filas).set_index("oficio")
ver(tabla_cer, "Errores de lectura de cada motor en los oficios escaneados")
decir(f"glm-ocr se equivocó en {num(tabla_cer['CER glm-ocr'].mean() * 100, 1)} de cada 100 caracteres y Tesseract en {num(tabla_cer['CER Tesseract'].mean() * 100, 1)}.")
# ✔ Control C4
assert (tabla_cer["CER glm-ocr"] < tabla_cer["CER Tesseract"]).all(), "C4: glm-ocr debe equivocarse menos que Tesseract en los 3 oficios."
CONTROLES.append("C4")
print("✔ C4 superado: glm-ocr lee mejor que Tesseract y el corte con 4096 quedó a la vista")

**Cómo leer esta salida**

- **Qué es cada fila:** Un oficio escaneado, leído por los dos motores.
- **Qué dice cada columna:** `CER`: errores por carácter; 0 es una lectura perfecta y 0,100 es un error cada diez caracteres. `segundos`: lo que tardó cada motor.
- **Qué mirar:** Qué motor tiene el CER más bajo en cada oficio y cuánto más tarda.
- **Qué se decide:** Para los escaneados se usa el motor con menos errores, aunque tarde más.

In [ ]:
# Ver E2: el oficio escaneado y el comienzo de lo que leyó cada motor
from PIL import Image

with pymupdf.open(DATOS / "02_Oficios" / "OFI-0002.pdf") as pdf:
    pagina = pdf[0].get_pixmap(dpi=60)
imagen = Image.frombytes("RGB", (pagina.width, pagina.height), pagina.samples)
display(imagen.crop((0, 0, imagen.width, int(imagen.height * 0.45))))
print("glm-ocr:\n" + "\n".join(texto_de(LEIDOS["OFI-0002.pdf"]).splitlines()[:8]))
print("\nTesseract:\n" + "\n".join(texto_de(LECTURAS_TESSERACT["OFI-0002"]).splitlines()[:8]))

## Pasos 3, 4 y 5: `llm/esquemas.py`, `llm/prompts.py` y `llm/extraer.py` (E3: la escalera del prompt, C5 y C6)

In [ ]:
# E3: el prompt completo de una póliza, tal como lo recibe el modelo (nivel 5: Rol, Contexto, Tarea, Formato y un ejemplo)
from mi_pipeline.llm.prompts import NIVELES, armar_mensajes, ver_prompt

texto_pol = texto_de(LEIDOS["POL-0001.pdf"])
mensajes_pol = armar_mensajes(texto_pol, "poliza", nivel=5)
ver_prompt(mensajes_pol)

**Cómo leer esta salida**

- **Qué se ve:** El prompt tal como se envía: el mensaje de sistema con ROL, CONTEXTO, TAREA y FORMATO; un ejemplo resuelto (usuario y asistente); y el DOCUMENTO. Los documentos aparecen recortados y la última línea de cada uno dice cuántos caracteres más tiene.
- **Qué mirar:** Cada parte responde una pregunta: ROL, quién responde; CONTEXTO, en qué situación; TAREA, qué hace; FORMATO, cómo entrega la respuesta.
- **Qué se decide:** Este texto es todo lo que recibe el modelo: lo que no está aquí, el modelo no lo sabe.

In [ ]:
# E3: la escalera del prompt: los mismos 13 archivos con 5 niveles, una parte más en cada nivel (C5 y C6)
from mi_pipeline.llm.extraer import extraer_con_llm

BANDEJA_E3 = [("05_Polizas/POL-0001.pdf", "poliza"), ("05_Polizas/POL-0002.pdf", "poliza"), ("04_Contratos/CON-0001.docx", "contrato"),
              ("04_Contratos/CON-0002.docx", "contrato"), ("02_Oficios/OFI-0001.pdf", "oficio"), ("02_Oficios/OFI-0002.pdf", "oficio"),
              ("02_Oficios/OFI-0003.pdf", "oficio"), ("03_Estados_Financieros/EFI-0001.pdf", "estado_financiero"), ("03_Estados_Financieros/EFI-0002.pdf", "estado_financiero"),
              ("08_Vouchers/VOU-0001.jpg", "voucher"), ("08_Vouchers/VOU-0002.jpg", "voucher"), ("06_Reclamos/REC-0001.eml", "reclamo"),
              ("06_Reclamos/REC-0002.eml", "reclamo")]
TECNICA = {1: "instrucción directa, sin ejemplos (zero-shot)", 2: "+ rol", 3: "+ contexto",
           4: "+ formato (el esquema escrito y en format)", 5: "+ un ejemplo resuelto (few-shot)"}
AGREGA = {1: "TAREA: campos, copiar tal cual, null", 2: "ROL: quién responde", 3: "CONTEXTO: la situación y cada tipo",
          4: "FORMATO: claves con su descripción", 5: "un documento del mismo tipo resuelto"}
TEXTOS_E3 = {}
for rel, tipo in BANDEJA_E3:
    ruta = DATOS / rel
    doc = LEIDOS.get(ruta.name) or leer_documento(ruta)
    if doc["pendientes"] and "ocr" not in doc:
        doc = aplicar_ocr(doc)                      # los 3 oficios ya llegan leídos del E2; aquí se leen los comprobantes y el adjunto
    LEIDOS[ruta.name] = doc
    TEXTOS_E3[ruta.stem] = (texto_de(doc), tipo)
ESPERADOS_E3 = sum(len(correcto_de(id_)) for id_ in TEXTOS_E3)
ESCALERA, filas = {}, []                            # ESCALERA[nivel][archivo]: lo que extrajo el LLM (el E5 lo revisa)
for nivel in NIVELES:
    ESCALERA[nivel] = {id_: extraer_con_llm(texto, tipo, nivel=nivel) for id_, (texto, tipo) in TEXTOS_E3.items()}
    bien = sum(aciertos(r["campos"], correcto_de(id_))[0] for id_, r in ESCALERA[nivel].items())
    filas.append({"nivel": nivel, "técnica": TECNICA[nivel], "qué se agrega": AGREGA[nivel], "correctos": bien, "de": ESPERADOS_E3,
                  "exactitud": round(bien / ESPERADOS_E3, 3), "tokens de entrada": sum(r["tokens_entrada"] for r in ESCALERA[nivel].values()),
                  "segundos": round(sum(r["segundos"] for r in ESCALERA[nivel].values()), 1)})
tabla_escalera = pd.DataFrame(filas).set_index("nivel")
ver(tabla_escalera, "La escalera del prompt: los mismos 13 archivos con 5 niveles")
_c = tabla_escalera["correctos"]
decir(f"con la sola Tarea acertó {_c[1]} de {ESPERADOS_E3}; con Rol y Contexto, {_c[3]}; con el Formato, {_c[4]}; con un ejemplo resuelto, {_c[5]}. El Formato sumó {_c[4] - _c[3]} aciertos y el ejemplo sumó {_c[5] - _c[4]}, con {num(tabla_escalera['tokens de entrada'][5] - tabla_escalera['tokens de entrada'][4])} tokens más.")
# ✔ Control C5
partes = [linea for linea in mensajes_pol[0]["content"].splitlines() if linea in ("ROL", "CONTEXTO", "TAREA", "FORMATO")]
assert partes == ["ROL", "CONTEXTO", "TAREA", "FORMATO"] and mensajes_pol[-1]["content"].startswith("DOCUMENTO"), "C5: el prompt de la póliza debe traer las cuatro partes y el documento."
assert len(tabla_escalera) == 5 and (tabla_escalera["tokens de entrada"] > 0).all(), "C5: la escalera debe medir los 5 niveles."
CONTROLES.append("C5")
# ✔ Control C6
exactitud_e3 = tabla_escalera["exactitud"]
assert exactitud_e3[4] > exactitud_e3[1] and exactitud_e3[5] >= exactitud_e3[4] - 0.02, "C6: el prompt completo (nivel 4) debe acertar más que la sola Tarea, y el ejemplo no debe bajar la exactitud."
CONTROLES.append("C6")
print(f"✔ C5 y C6 superados: {pct(exactitud_e3[1])} con la sola Tarea -> {pct(exactitud_e3[4])} con Rol, Contexto, Tarea y Formato -> {pct(exactitud_e3[5])} con un ejemplo resuelto")

**Cómo leer esta salida**

- **Qué es cada fila:** Un nivel del prompt: cada nivel agrega una parte al anterior y usa los mismos 13 archivos.
- **Qué dice cada columna:** `técnica` y `qué se agrega`: lo nuevo de ese nivel. `correctos`: campos bien extraídos, de los 144 de la columna `de`. `exactitud`: el mismo dato en porcentaje. `tokens de entrada`: cuánto leyó el modelo en total; un token es un pedazo de palabra de unas 3 o 4 letras, y más tokens significan más costo. `segundos`: el tiempo del nivel; el del nivel 1 incluye la carga del modelo.
- **Qué mirar:** En qué fila sube más `correctos` y cuántos tokens cuesta esa subida.
- **Qué se decide:** Una parte se deja si lo que suma en aciertos justifica lo que cuesta en tokens. La muestra es de 13 archivos y una sola corrida: un campo de diferencia entre dos niveles puede ser variación entre corridas.

In [ ]:
# E3: qué corrige cada parte del prompt: los errores del nivel anterior que desaparecen, y la exactitud por tipo
def errores(nivel):
    """Los campos mal extraídos en un nivel: «archivo campo» -> detalle del fallo."""
    return {f"{id_} {fallo.split(':')[0]}": f"{id_} {fallo}" for id_, r in ESCALERA[nivel].items() for fallo in aciertos(r["campos"], correcto_de(id_))[2]}


def por_tipo(nivel):
    """Correctos / esperados por tipo de documento en un nivel."""
    cuenta = {}
    for id_, r in ESCALERA[nivel].items():
        bien, total, _ = aciertos(r["campos"], correcto_de(id_))
        previo = cuenta.get(TEXTOS_E3[id_][1], (0, 0))
        cuenta[TEXTOS_E3[id_][1]] = (previo[0] + bien, previo[1] + total)
    return {tipo: f"{bien}/{total}" for tipo, (bien, total) in cuenta.items()}


print("Errores con la sola Tarea (nivel 1):\n  " + "\n  ".join(errores(1).values()))
CORRIGE = {}
for nivel in list(NIVELES)[1:]:
    antes, ahora = errores(nivel - 1), errores(nivel)
    CORRIGE[nivel] = sorted(set(antes) - set(ahora))
    nuevos = sorted(set(ahora) - set(antes))
    print(f"Nivel {nivel} ({NIVELES[nivel]}): desaparecen {len(CORRIGE[nivel])}: {', '.join(CORRIGE[nivel]) or 'ninguno'}"
          + (f" | aparecen: {', '.join(nuevos)}" if nuevos else ""))
print("Errores que quedan en el nivel 5:", "; ".join(errores(5).values()) or "ninguno")
tabla_tipos_e3 = pd.DataFrame({f"nivel {nivel}": por_tipo(nivel) for nivel in NIVELES})
ver(tabla_tipos_e3.rename_axis("tipo"), "Campos correctos por tipo de documento y nivel")

**Cómo leer esta salida**

- **Qué se ve:** Los errores del nivel 1, los que desaparecen en cada nivel, los que aparecen, los que quedan en el nivel 5 y una tabla por tipo de documento.
- **Cómo se lee un error:** «OFI-0002 entidad: 'X' en lugar de 'Y'» significa que, en ese archivo y ese campo, el modelo escribió X y lo correcto era Y. Sin el detalle, «OFI-0002 entidad» nombra el archivo y el campo.
- **Qué dice la tabla:** Cada columna es un nivel; cada dato, campos correctos sobre campos esperados de ese tipo de documento.
- **Qué se decide:** Si un tipo de documento sigue con errores en el nivel 5, se mejora la parte del prompt que le corresponde o se agrega un validador.

In [ ]:
# E3: prompt propio para los oficios. Se hacen dos cambios, uno a la vez, y se ejecuta de nuevo esta celda después de cada uno
# Cambio 1: reescribir MI_CONTEXTO, entre las comillas: qué documento es, quién lo emite y a quién se dirige
# Cambio 2: agregar en MiOficio una línea para el campo plazo_dias, como la de firmante: plazo_dias: float | None = Field(..., description="...")
from pydantic import BaseModel, Field

MI_CONTEXTO = "Documento que llega a la mesa de partes."


class MiOficio(BaseModel):
    numero_oficio: str | None = Field(..., description="número del oficio")
    fecha: str | None = Field(..., description="fecha del oficio")
    entidad: str | None = Field(..., description="entidad")
    emisor: str | None = Field(..., description="quién emite")
    firmante: str | None = Field(..., description="nombre y apellidos de quien firma")


# Desde aquí no se cambia nada: se extraen los 3 oficios con este prompt y se mide
OFICIOS = ["OFI-0001", "OFI-0002", "OFI-0003"]
ver_prompt(armar_mensajes(TEXTOS_E3[OFICIOS[0]][0], "oficio", 4, MiOficio, contexto=MI_CONTEXTO), max_doc=150)
columnas, errados, bien, total, tokens, segundos = {}, set(), 0, 0, 0, 0.0
for id_ in OFICIOS:
    r = extraer_con_llm(TEXTOS_E3[id_][0], "oficio", esquema=MiOficio, nivel=4, contexto=MI_CONTEXTO)
    correcto, columna = correcto_de(id_), {}
    for campo, valor in r["campos"].items():
        if campo not in correcto:
            columna[campo] = f"{como_texto(valor)} (no se mide)"
        elif es_correcto(valor, correcto[campo]):
            bien, total = bien + 1, total + 1
            columna[campo] = f"✔ {como_texto(valor)}"
        else:
            total += 1
            errados.add(campo)
            columna[campo] = f"✘ {como_texto(valor)} (lo correcto: {como_texto(correcto[campo])})"
    columnas[id_] = columna
    tokens, segundos = tokens + r["tokens_entrada"], segundos + r["segundos"]
ver(pd.DataFrame(columnas).rename_axis("campo"), "Lo que extrajo el prompt propio")
if "MIS_INTENTOS" not in globals():
    MIS_INTENTOS = []
MIS_INTENTOS.append({"intento": len(MIS_INTENTOS) + 1, "contexto propio": MI_CONTEXTO, "campos": len(MiOficio.model_fields), "correctos": bien, "de": total,
                     "exactitud": round(bien / total, 3), "tokens de entrada": tokens, "segundos": round(segundos, 1)})
ver(pd.DataFrame(MIS_INTENTOS), "Intentos del participante")
decir(f"este prompt acertó {bien} de {total} campos; " + (f"los errores están en estos campos: {', '.join(sorted(errados))}." if errados else "sin errores en estos 3 oficios."))
faltan = []
if MI_CONTEXTO == "Documento que llega a la mesa de partes.":
    faltan.append("reescribir MI_CONTEXTO")
if "plazo_dias" not in MiOficio.model_fields:
    faltan.append("agregar el campo plazo_dias en MiOficio")
print("✔ Los dos cambios están hechos: Contexto propio y campo nuevo" if not faltan else "✘ Falta: " + " y ".join(faltan) + " (se cambia arriba y se ejecuta de nuevo esta celda)")

**Cómo leer esta salida**

- **Qué se ve:** El prompt armado con el Contexto y el esquema de la celda; una tabla con lo que extrajo; y «Intentos del participante», con una fila por cada vez que se ejecutó la celda.
- **Qué dice la tabla:** Cada fila es un campo y cada columna, un oficio. ✔ es un acierto; ✘ es un error, con el valor correcto entre paréntesis.
- **Qué mirar:** Los campos `entidad` y `emisor`: con un Contexto pobre el modelo confunde a quien envía con quien recibe. Después del cambio 1, ¿siguen los ✘? Después del cambio 2, ¿aparece `plazo_dias` con el valor correcto?
- **Qué se decide:** Si un campo sigue mal, se mejora la parte del prompt que lo explica (el Contexto o la descripción del campo en `MiOficio`) y se mide de nuevo.

## Paso 6: `llm/trozos.py` (E4: dividir y combinar, C7 y C8)

In [ ]:
# E4: un contrato de 14 páginas con el contexto de serie (4096) y con el del YAML (12288) (C7)
from mi_pipeline.llm.trozos import cabe, estimar_tokens, map_reduce, por_pagina, ventana

contrato = leer_documento(DATOS / "04_Contratos" / "CON-0021.pdf")
texto_con = texto_de(contrato)
correcto_con = correcto_de("CON-0021")
print(f"CON-0021: {len(contrato['partes'])} páginas, {num(len(texto_con))} caracteres, unos {num(estimar_tokens(texto_con))} tokens."
      f" ¿Cabe en un contexto de 4096? {si_no(cabe(texto_con, 4096))}. ¿En uno de 12288? {si_no(cabe(texto_con, 12288))}.")
filas = []
for contexto in (4096, 12288):
    r = extraer_con_llm(texto_con, "contrato", num_ctx=contexto)
    ok, n, fallos = aciertos(r["campos"], correcto_con)
    filas.append({"contexto": contexto, "tokens que leyó": r["tokens_entrada"], "correctos": ok, "esperados": n, "segundos": r["segundos"],
                  "campos perdidos o errados": ", ".join(f.split(":")[0] for f in fallos) or "ninguno"})
    if contexto == 12288:
        ENTERO_CON = r
tabla_contexto = pd.DataFrame(filas).set_index("contexto")
ver(tabla_contexto, "El mismo contrato con dos contextos")
_corto, _largo = tabla_contexto.loc[4096], tabla_contexto.loc[12288]
decir(f"el contrato tiene unos {num(estimar_tokens(texto_con))} tokens. Con contexto 4096 el modelo leyó {num(_corto['tokens que leyó'])} y acertó {_corto['correctos']} de {_corto['esperados']}, sin ningún mensaje de error. Con 12288 leyó {num(_largo['tokens que leyó'])} y acertó {_largo['correctos']} de {_largo['esperados']}.")
# ✔ Control C7
assert tabla_contexto.loc[4096, "correctos"] < tabla_contexto.loc[12288, "correctos"], "C7: con 4096 deben perderse campos que con 12288 no."
assert tabla_contexto.loc[4096, "tokens que leyó"] < estimar_tokens(texto_con), "C7: con 4096 el modelo lee menos tokens de los que tiene el contrato."
CONTROLES.append("C7")
print("✔ C7 superado: con 4096 el contrato se cortó sin error y con 12288 llegó completo")

**Cómo leer esta salida**

- **Qué es cada fila:** El mismo contrato con un contexto distinto. El contexto es cuánto texto cabe a la vista del modelo de una sola vez, medido en tokens (un token es un pedazo de palabra de unas 3 o 4 letras).
- **Qué dice cada columna:** `tokens que leyó`: lo que el modelo leyó en realidad. `correctos` y `esperados`: campos bien extraídos y campos que había que extraer. `segundos`: lo que tardó. `campos perdidos o errados`: los campos que salieron mal.
- **Qué mirar:** Si `tokens que leyó` es menor que los tokens del contrato (primera línea), el modelo no vio todo el documento. No aparece ningún error: el modelo responde con lo que alcanzó a leer.
- **Qué se decide:** Antes de extraer se comprueba si el documento cabe; si no cabe, se aumenta el contexto o se divide en trozos.

In [ ]:
# E4: dividir y combinar (map-reduce): el contrato entero, por página y en ventanas, con el mismo prompt en cada trozo (C8)
trozos_pagina = por_pagina(contrato)
trozos_ventana = ventana(texto_con)
estrategias = {"entero (12288)": {"campos": ENTERO_CON["campos"], "llamadas": 1, "segundos": ENTERO_CON["segundos"]},
               "por página + votación": map_reduce(trozos_pagina, "contrato"),
               "ventanas + votación": map_reduce(trozos_ventana, "contrato")}
filas = []
for nombre, r in estrategias.items():
    ok, n, fallos = aciertos(r["campos"], correcto_con)
    filas.append({"estrategia": nombre, "llamadas": r["llamadas"], "correctos": ok, "esperados": n, "segundos": r["segundos"],
                  "fallos": "; ".join(fallos) or "ninguno"})
tabla_estrategias = pd.DataFrame(filas).set_index("estrategia")
ver(tabla_estrategias, "Tres formas de leer el mismo contrato")
ver(pd.DataFrame([{"página": k, "campos que trajo": ", ".join(c for c, v in p.items() if v is not None) or "ninguno"}
                  for k, p in enumerate(estrategias["por página + votación"]["por_trozo"], 1)]), "Qué campos trajo cada página")
decir("; ".join(f"{nombre}: {f['correctos']} de {f['esperados']} campos, en {num(f['segundos'], 1)} s (llamadas: {f['llamadas']})" for nombre, f in tabla_estrategias.iterrows()) + ".")
# ✔ Control C8
assert tabla_estrategias.loc["por página + votación", "llamadas"] > tabla_estrategias.loc["ventanas + votación", "llamadas"] > 1, "C8: por página hace más llamadas que las ventanas."
CONTROLES.append("C8")
print("✔ C8 superado: tres estrategias comparadas en exactitud, llamadas y segundos")

**Cómo leer esta salida**

- **Qué es cada fila:** Una forma de leer el mismo contrato: entero, página por página o en ventanas de texto.
- **Qué dice cada columna:** `llamadas`: cuántas veces se preguntó al modelo. `correctos` y `esperados`: campos bien extraídos y campos que había que extraer. `segundos`: la suma de las llamadas. `fallos`: el campo, lo que salió y lo correcto. La segunda tabla dice qué campos encontró cada página.
- **Qué mirar:** La votación junta lo que encontró cada trozo: gana el valor que más trozos devolvieron. Cada llamada lee menos texto; por eso, más llamadas no siempre son más segundos.
- **Qué se decide:** Entero si cabe, porque el modelo ve todo el documento de una vez; ventanas si no cabe; por página cuando importa saber de qué página salió cada dato.

In [ ]:
# E4: un expediente de 64 páginas, mixto (nativo y escaneado), que no cabe en el contexto
expediente = leer_documento(DATOS / "13_Expedientes" / "EXP-0002.pdf")
texto_exp = texto_de(expediente)
correcto_exp = correcto_de("EXP-0002")
print(f"EXP-0002: {len(expediente['partes'])} páginas; {len(expediente['pendientes'])} sin capa de texto "
      f"({', '.join(p['origen'].split()[-1] for p in expediente['pendientes'])}); unos {num(estimar_tokens(texto_exp))} tokens con lo que trae el PDF.")
entero_exp = extraer_con_llm(texto_exp, "expediente")
ventanas_exp = map_reduce(ventana(texto_exp, tamano=35000, solape=2000), "expediente")
filas = []
for nombre, r, tokens in (("entero (12288)", entero_exp, entero_exp["tokens_entrada"]), ("ventanas de 35.000 + votación", ventanas_exp, None)):
    ok, n, fallos = aciertos(r["campos"], correcto_exp)
    filas.append({"estrategia": nombre, "llamadas": r.get("llamadas", 1), "tokens que leyó": tokens, "correctos": ok, "esperados": n,
                  "segundos": r["segundos"], "fallos": "; ".join(fallos) or "ninguno"})
tabla_expediente = pd.DataFrame(filas).set_index("estrategia")
ver(tabla_expediente, "Un expediente que no cabe, leído entero y en ventanas", vacio="no aplica")
_entero, _ventanas = tabla_expediente.iloc[0], tabla_expediente.iloc[1]
decir(f"el expediente tiene unos {num(estimar_tokens(texto_exp))} tokens. Entero, el modelo leyó {num(_entero['tokens que leyó'])} y acertó {_entero['correctos']} de {_entero['esperados']}; en ventanas acertó {_ventanas['correctos']} de {_ventanas['esperados']} (llamadas: {_ventanas['llamadas']}).")

**Cómo leer esta salida**

- **Qué es cada fila:** Un expediente largo, leído entero o en ventanas.
- **Qué dice cada columna:** Las mismas de la tabla anterior. `tokens que leyó` se mide solo en la lectura entera; en las ventanas dice «no aplica», porque son varias llamadas.
- **Qué mirar:** Los `tokens que leyó` en la fila «entero» frente a los tokens del expediente (primera línea): el modelo solo vio una parte.
- **Qué se decide:** Un documento largo se procesa por partes y solo con las páginas que hacen falta para la pregunta.

## Paso 7: `calidad/alucinaciones.py` (E5, C9 y C10)

In [ ]:
# E5: ¿el valor figura en el texto y pasa los validadores? El detector sobre todo lo que extrajo la escalera del E3 (C9)
from mi_pipeline.calidad.alucinaciones import a_revisar, aparece, validar_campo

filas, resumen = [], []
for nivel, salidas in ESCALERA.items():
    entregados = errados = 0
    for id_, r in salidas.items():
        texto, correcto = TEXTOS_E3[id_][0], correcto_de(id_)
        errados += len(aciertos(r["campos"], correcto)[2])
        for campo, valor in r["campos"].items():
            if valor is None:
                continue
            entregados += 1
            causa = a_revisar(campo, valor, texto, r["campos"])
            if causa:
                filas.append({"nivel": nivel, "archivo": id_, "campo": campo, "valor": valor, "causa": causa,
                              "valor correcto": correcto.get(campo), "¿era correcto?": es_correcto(valor, correcto.get(campo))})
    marcados = [f for f in filas if f["nivel"] == nivel]
    resumen.append({"nivel": nivel, "valores entregados": entregados, "marcados": len(marcados),
                    "marcados que eran errores": sum(not f["¿era correcto?"] for f in marcados), "errores del nivel": errados})
tabla_resumen_e5 = pd.DataFrame(resumen).set_index("nivel")
tabla_marcados = pd.DataFrame(filas, columns=["nivel", "archivo", "campo", "valor", "causa", "valor correcto", "¿era correcto?"])
ver(tabla_resumen_e5, "Por nivel: qué marcó el detector")
ver(tabla_marcados, "Cada valor marcado, con su causa")
_n1, _n5 = tabla_resumen_e5.loc[1], tabla_resumen_e5.loc[5]
decir(f"con la sola Tarea: valores marcados, {_n1['marcados']}; errores del nivel, {_n1['errores del nivel']}. Con el prompt completo: valores marcados, {_n5['marcados']}; errores del nivel, {_n5['errores del nivel']}.")
REALES = {causa: next((f for f in filas if f["causa"] == causa and not f["¿era correcto?"]), None)
          for causa in ("no figura en el texto", "no pasa el validador")}      # un ejemplo real de cada causa, el del nivel más bajo
for causa, ejemplo in REALES.items():
    print(f"Ejemplo real («{causa}»):", f"{ejemplo['archivo']}, campo {ejemplo['campo']}: el modelo escribió «{como_texto(ejemplo['valor'])}» y lo correcto era «{como_texto(ejemplo['valor correcto'])}»." if ejemplo else "ninguno en esta corrida")
# ✔ Control C9 (con un caso escrito a mano, para no depender de lo que responda el modelo)
texto_efi = TEXTOS_E3["EFI-0001"][0]
assert not aparece("millones de soles", texto_efi) and aparece(39317.0, texto_efi), "C9: el detector debe marcar una unidad que no está en el texto y aceptar 39317 (39,317 en el texto)."
CONTROLES.append("C9")
print("✔ C9 superado: el detector marca lo que no figura en el texto")

**Cómo leer esta salida**

- **Qué es cada fila:** En la primera tabla, un nivel del prompt. En la segunda, un valor que el detector puso en duda.
- **Qué dice cada columna:** `valores entregados`: campos con dato. `marcados`: los que el detector puso en duda. `marcados que eran errores`: los marcados que estaban mal. `errores del nivel`: todos los errores, con marca o sin ella. `causa`: por qué se marcó. `¿era correcto?`: si el valor marcado era bueno a pesar de todo (por ejemplo, una letra corregida).
- **Qué mirar:** `marcados` es menor que `errores del nivel`: el detector no ve un valor equivocado que sí figura en el texto ni un dato que quedó vacío.
- **Qué se decide:** Un valor que no figura en el texto o que no pasa un validador no entra solo: va a revisión humana. Como el detector no atrapa todos los errores, se revisa además una muestra aunque la lista salga vacía.

In [ ]:
# E5: validadores de coherencia con la sola Tarea (nivel 1) y con el prompt completo (nivel 5): TEA y TCEA, emisor y firmante (C10)
casos = [("04_Contratos/CON-0022.pdf", "contrato", ("tea", "tcea")), ("02_Oficios/OFI-0004.pdf", "oficio", ("emisor", "firmante")),
         ("04_Contratos/CON-0001.docx", "contrato", ("fecha_contrato",)), ("04_Contratos/CON-0003.docx", "contrato", ("fecha_contrato",))]
filas = []
for rel, tipo, campos in casos:
    ruta = DATOS / rel
    doc = LEIDOS.get(ruta.name) or leer_documento(ruta)
    doc = aplicar_ocr(doc) if doc["pendientes"] and "ocr" not in doc else doc
    LEIDOS[ruta.name] = doc
    for nivel in (1, 5):
        r = ESCALERA[nivel].get(ruta.stem) or extraer_con_llm(texto_de(doc), tipo, nivel=nivel)    # lo que ya extrajo el E3 no se repite
        for campo in campos:
            valor = r["campos"][campo]
            filas.append({"archivo": ruta.stem, "nivel": nivel, "campo": campo, "valor del LLM": valor, "valor correcto": correcto_de(ruta.stem)[campo],
                          "causa para revisar": a_revisar(campo, valor, texto_de(doc), r["campos"]) or "se acepta"})
tabla_validadores = pd.DataFrame(filas)
ver(tabla_validadores, "Los validadores con la sola Tarea (nivel 1) y con el prompt completo (nivel 5)")
_objetados = tabla_validadores[tabla_validadores["causa para revisar"] != "se acepta"].groupby("nivel").size()
decir(f"valores que el validador objetó: {_objetados.get(1, 0)} con la sola Tarea y {_objetados.get(5, 0)} con el prompt completo.")
# ✔ Control C10
assert not validar_campo("tea", 25.0, {"tcea": 20.0}) and validar_campo("tea", 20.0, {"tcea": 25.0}), "C10: una TEA mayor o igual que la TCEA se rechaza."
assert not validar_campo("emisor", "Ana Ruiz Soto", {}) and validar_campo("emisor", "Intendencia DEMO de Riesgos", {}), "C10: un emisor sin «Intendencia» se rechaza."
CONTROLES.append("C10")
print("✔ C10 superado: los validadores rechazan una TEA mayor que la TCEA y un emisor que no es una intendencia")

**Cómo leer esta salida**

- **Qué es cada fila:** Un campo de un documento, con la sola Tarea (nivel 1) o con el prompt completo (nivel 5).
- **Qué dice cada columna:** `valor del LLM`: lo que escribió el modelo; «vacío» significa que no dio ningún dato. `valor correcto`: la respuesta real. `causa para revisar`: «no pasa el validador» indica que el valor no cumple una regla de coherencia, como un emisor que no es una intendencia.
- **Qué mirar:** La misma fila en los dos niveles. El texto «se acepta» solo dice que el validador no objetó nada: un valor vacío se acepta aunque el dato exista, y es un dato perdido.
- **Qué se decide:** Los validadores de coherencia se escriben una vez y revisan los valores entregados, con cualquier prompt. No recuperan lo que el modelo dejó vacío.

## Paso 8: `extraccion/hibrido.py` (E6: el LLM como respaldo, C11)

In [ ]:
# E6: la cadena completa Ingesta -> OCR -> PLN -> LLM (técnica: el LLM como respaldo) sobre una bandeja de 13 archivos (C11)
from mi_pipeline.extraccion.hibrido import evaluar_bandeja, procesar_bandeja

BANDEJA_E6 = [("05_Polizas/POL-0004.pdf", "poliza"), ("05_Polizas/POL-0005.pdf", "poliza"), ("04_Contratos/CON-0022.pdf", "contrato"),
              ("04_Contratos/CON-0002.docx", "contrato"), ("04_Contratos/CON-0005.docx", "contrato"), ("02_Oficios/OFI-0004.pdf", "oficio"),
              ("02_Oficios/OFI-0005.pdf", "oficio"), ("03_Estados_Financieros/EFI-0032.pdf", "estado_financiero"), ("08_Vouchers/VOU-0004.jpg", "voucher"),
              ("08_Vouchers/VOU-0005.png", "voucher"), ("06_Reclamos/REC-0001.eml", "reclamo"), ("06_Reclamos/REC-0002.eml", "reclamo"),
              ("07_Tarifarios/TAR-0003.html", "tarifario")]
resultados_e6 = procesar_bandeja([(DATOS / rel, tipo) for rel, tipo in BANDEJA_E6])
evaluacion_e6 = evaluar_bandeja(resultados_e6, {Path(rel).stem: correcto_de(Path(rel).stem) for rel, _ in BANDEJA_E6}, es_correcto)
tabla_fases_e6 = pd.DataFrame(evaluacion_e6["por_fase"]).set_index("fase")
ver(tabla_fases_e6, "Exactitud después de cada fase")
ver(pd.DataFrame(evaluacion_e6["por_tipo"]).set_index("tipo"), "Campos correctos por tipo de documento")
revisar_e6 = pd.DataFrame([{"archivo": r["archivo"], **x} for r in resultados_e6 for x in r.get("revisar", [])])
ver(revisar_e6, "Lista «revisar»: valores del modelo que no entran solos")
ambiguos_e6 = pd.DataFrame([{"archivo": r["archivo"], **x, "valor correcto": correcto_de(Path(r["archivo"]).stem).get(x["campo"])} for r in resultados_e6 for x in r.get("ambiguos", [])],
                           columns=["archivo", "campo", "regla", "modelo", "valor correcto"])
ver(ambiguos_e6, "Lista «ambiguos»: la regla y el modelo no coincidieron y entró el valor del modelo")
# ✔ Control C11
exactitud = tabla_fases_e6["exactitud"]
assert exactitud["1_ingesta"] < exactitud["2_ocr"] < exactitud["3_pln"] < exactitud["4_llm"], "C11: la exactitud debe subir en cada fase, sin empates."
CONTROLES.append("C11")
print(f"✔ C11 superado: {pct(exactitud['1_ingesta'])} -> {pct(exactitud['2_ocr'])} -> {pct(exactitud['3_pln'])} -> {pct(exactitud['4_llm'])}")
segundos_e6 = tabla_fases_e6["segundos"]
llamadas_e6 = sum(r.get("llamo_modelo", False) for r in resultados_e6)
decir(f"la cadena pasó de {pct(exactitud['1_ingesta'])} a {pct(exactitud['4_llm'])}. Se llamó al modelo en {llamadas_e6} de {len(resultados_e6)} archivos; campos ambiguos resueltos: {len(ambiguos_e6)}. "
      f"Como comparación, el LLM solo, con el mismo prompt y el mismo texto del OCR, llegó a {pct(exactitud['solo_llm'])}. Las reglas (fases 1 y 3) tardaron {num(segundos_e6[['1_ingesta', '3_pln']].sum(), 1)} s, el OCR {num(segundos_e6['2_ocr'], 1)} s y el modelo {num(segundos_e6['4_llm'], 1)} s.")

**Cómo leer esta salida**

- **Qué es cada fila:** En la primera tabla, una fase de la cadena, en orden; `solo_llm` no es una fase: es la comparación con el modelo sin las fases anteriores.
- **Qué dice cada columna:** `correctos` y `esperados`: campos bien extraídos y campos que había que extraer. `exactitud`: el mismo dato en porcentaje. `precision`: de lo que se entregó, cuánto está bien. `exhaustividad`: de lo que existe, cuánto se entregó bien. `f1`: el equilibrio entre las dos. `segundos`: lo que tardó la fase.
- **Qué dicen las listas:** «revisar»: valores del modelo que no figuran en el texto o no pasan un validador; no entran. «ambiguos»: campos que dedujo una regla sin un rótulo que la respalde, en los que el modelo dio otro valor comprobable; entra el del modelo y queda anotado.
- **Qué mirar y qué se decide:** En qué fase sube más la exactitud y si sube en cada una: se invierte en la fase que más suma para cada tipo de documento. Lo que queda en «revisar» lo decide una persona; lo que aparece en «ambiguos» dice qué regla conviene corregir.

In [ ]:
# Ver E6: un mismo oficio escaneado, campo por campo, después de cada fase
oficio_e6 = next(r for r in resultados_e6 if r["archivo"] == "OFI-0004.pdf")
correcto_e6 = correcto_de("OFI-0004")
vista = pd.DataFrame({fase: oficio_e6["fases"][fase] for fase in ("1_ingesta", "2_ocr", "3_pln", "4_llm", "solo_llm")})
vista["correcto"] = pd.Series(correcto_e6)
ver(vista.rename_axis("campo"), "El oficio OFI-0004 campo por campo, después de cada fase")

**Cómo leer esta salida**

- **Qué es cada fila:** Un campo del oficio.
- **Qué dice cada columna:** Lo que había en ese campo después de cada fase; «vacío» significa que todavía no tenía dato. La última columna es el valor correcto.
- **Qué mirar:** De izquierda a derecha, en qué fase aparece cada dato.
- **Qué se decide:** Se ve qué aporta cada fase y qué valor del modelo quedó fuera.

## Práctica evaluada: bandeja mixta de la mesa de partes DEMO

In [ ]:
# Práctica evaluada: la cadena sobre una bandeja nueva de 10 archivos; tabla por fase, por tipo y lista «revisar»
BANDEJA_PRACTICA = [("05_Polizas/POL-0011.pdf", "poliza"), ("05_Polizas/POL-0012.pdf", "poliza"), ("04_Contratos/CON-0023.pdf", "contrato"),
              ("04_Contratos/CON-0006.docx", "contrato"), ("02_Oficios/OFI-0006.pdf", "oficio"), ("02_Oficios/OFI-0007.pdf", "oficio"),
              ("03_Estados_Financieros/EFI-0033.pdf", "estado_financiero"), ("08_Vouchers/VOU-0006.jpg", "voucher"), ("06_Reclamos/REC-0003.eml", "reclamo"),
              ("07_Tarifarios/TAR-0004.html", "tarifario")]
resultados_p = procesar_bandeja([(DATOS / rel, tipo) for rel, tipo in BANDEJA_PRACTICA])
errores_p = [f"{r['archivo']}: {r['error']}" for r in resultados_p if "error" in r]
print(f"Archivos procesados: {len(resultados_p) - len(errores_p)} de {len(BANDEJA_PRACTICA)}; con error: {errores_p or 'ninguno'}")
evaluacion_p = evaluar_bandeja(resultados_p, {Path(rel).stem: correcto_de(Path(rel).stem) for rel, _ in BANDEJA_PRACTICA}, es_correcto)
tabla_fases_p = pd.DataFrame(evaluacion_p["por_fase"]).set_index("fase")
tabla_tipos_p = pd.DataFrame(evaluacion_p["por_tipo"]).set_index("tipo")
revisar_p = pd.DataFrame([{"archivo": r["archivo"], **x} for r in resultados_p for x in r.get("revisar", [])], columns=["archivo", "campo", "valor", "causa"])
ambiguos_p = pd.DataFrame([{"archivo": r["archivo"], **x, "valor correcto": correcto_de(Path(r["archivo"]).stem).get(x["campo"])} for r in resultados_p for x in r.get("ambiguos", [])],
                          columns=["archivo", "campo", "regla", "modelo", "valor correcto"])
ver(tabla_fases_p, "Exactitud después de cada fase")
ver(tabla_tipos_p, "Campos correctos por tipo de documento")
ver(revisar_p, "Lista «revisar»: valores del modelo que no entran solos")
ver(ambiguos_p, "Lista «ambiguos»: la regla y el modelo no coincidieron y entró el valor del modelo")
tabla_fases_p.to_csv(SALIDAS / "practica_s05_fases.csv", encoding="utf-8")
tabla_tipos_p.to_csv(SALIDAS / "practica_s05_tipos.csv", encoding="utf-8")
revisar_p.to_csv(SALIDAS / "practica_s05_revisar.csv", index=False, encoding="utf-8")
UMBRAL_PRACTICA = 0.80
final = tabla_fases_p.loc["4_llm", "exactitud"]
e_p = tabla_fases_p["exactitud"]
sube = bool(e_p["1_ingesta"] < e_p["2_ocr"] < e_p["3_pln"] < e_p["4_llm"])   # sube en cada fase, sin empates
print(("✔" if final >= UMBRAL_PRACTICA and sube else "✘"), f"exactitud final {pct(final)} (umbral {pct(UMBRAL_PRACTICA)}); sube en cada fase: {si_no(sube)}")
print("Tablas guardadas en", SALIDAS, "para copiarlas a la plantilla de la práctica")

**Cómo leer esta salida**

- **Qué se ve:** Lo mismo que en el E6, con una bandeja nueva: la tabla por fase, la tabla por tipo y las listas «revisar» y «ambiguos».
- **Qué mirar:** La fila `4_llm` frente al umbral y frente a `solo_llm`.
- **Qué se hace con esto:** Las tablas guardadas en `salidas` se copian a la plantilla y las preguntas se responden con estos números.

## Reto extra (opcional)

In [ ]:
# Reto extra (opcional): la misma cadena con Tesseract en lugar de glm-ocr en los 6 archivos que necesitan OCR
CORRER_RETO_EXTRA = False        # True para correrlo (tarda alrededor de un minuto)
if CORRER_RETO_EXTRA:
    con_ocr = [(rel, tipo) for rel, tipo in BANDEJA_E6 if any(r["archivo"] == Path(rel).name and r["paginas_ocr"] for r in resultados_e6)]
    con_tesseract = procesar_bandeja([(DATOS / rel, tipo) for rel, tipo in con_ocr], motor_ocr="tesseract")
    con_glm = [r for r in resultados_e6 if r["archivo"] in {Path(rel).name for rel, _ in con_ocr}]
    correctos_ocr = {Path(rel).stem: correcto_de(Path(rel).stem) for rel, _ in con_ocr}
    comparacion = pd.DataFrame({"glm-ocr": pd.DataFrame(evaluar_bandeja(con_glm, correctos_ocr, es_correcto)["por_fase"]).set_index("fase")["exactitud"],
                                "Tesseract": pd.DataFrame(evaluar_bandeja(con_tesseract, correctos_ocr, es_correcto)["por_fase"]).set_index("fase")["exactitud"]})
    ver(comparacion, "La cadena con los dos motores de OCR: exactitud por fase")
else:
    print("Reto extra omitido: se cambia CORRER_RETO_EXTRA a True para comparar los dos motores de OCR en la cadena.")

## Cierre

In [ ]:
# ✔ Cierre: los controles, las pruebas y el tiempo del cuaderno
esperados = ["C0", "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11"]
faltan = [c for c in esperados if c not in CONTROLES]
assert not faltan, f"Faltan controles por superar: {faltan}"
assert correr_pruebas(), "Alguna prueba de mi_pipeline falla: se lee el mensaje de pytest y se corrige el archivo que indica."
print("✔ Controles superados:", ", ".join(CONTROLES))
print(f"Tiempo del cuaderno: {(time.perf_counter() - INICIO) / 60:.1f} minutos")
print("Siguiente: guardar el avance (commit), etiquetar y subir la rama trabajo-s05, como indica la guía.")